<img src="../img/logo.jpeg" width= 200px>   

# 📓 Notebook 2 - Ejercicio: construye tu propio Agente con `tools`

Este notebook es una **propuesta práctica** para poner en juego lo aprendido en los notebooks anteriores:

- `0_AgentesIA__Agente_Simple_Langchain` → creación de un agente y de herramientas (`@tool`).
- `1_AgentesIA_Agentes_LangGraph` → flujo de razonamiento, herramientas y respuestas.

**El objetivo** es que construyas **un agente con varias `tools`** capaz de resolver **una tarea concreta**.
Como la tarea no es única, más abajo se te ofrecen **5 opciones distintas**: elige **una** y construye tu agente alrededor de ella. 🎯

> 💡 Regla de oro: *una `tool` no tiene por qué hacer cálculos complejos; basta con que devuelva información útil (siempre como texto `str`) que el agente no pueda obtener por sí solo de forma fiable.*

------


## 🗂️ Índice   
1.- ⚙️ Pasos previos (API keys e instalación).     
2.- 🔀 Importar librerías.      
3.- 🧩 Conectar con el LLM (motor del agente).          
4.- 🤖 Repaso: ¿qué es un agente? (patrón ReAct).     
5.- 🎯 El ejercicio: elige tu tarea entre 5 opciones.     
6.- 🔧 Implementa las `tools` de tu opción.     
7.- 🏗️ Construye el agente.     
8.- 🔍 Prueba tu agente.     
9.- 🧠 Reto extra: historial de conversación.     
10.- ✅ Rúbrica de evaluación.

-------


## 1.- ⚙️ Pasos previos

### 🔑 Configuración de las API keys necesarias

Ejecuta el siguiente bloque después de introducir tu API key de **GROQ** (es la única imprescindible para este ejercicio).


In [ ]:
# =====================================================================================
# 🔑  Introduce tus API keys en la linea correspondiente antes de ejecutar el notebook
# =====================================================================================
import os

# Carga tu API key de GROQ en la siguiente variable.
# Puedes obtenerla (gratis) en https://console.groq.com/keys
GROQ_API_KEY = ""  # 👈 Pon aquí tu token de GROQ

# (Opcional) API key de Hugging Face, por si quieres experimentar con modelos de HF.
# Puedes obtenerla en https://huggingface.co/settings/tokens
HF_API_KEY = ""  # 👈 (opcional)

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_API_KEY

print("✅ ¡API keys configuradas y listas para usar!")


### 📚 Instalación de librerías

Instalamos las librerías necesarias (sólo la primera vez). Incluye `wikipedia` por si eliges la **opción C (tutor educativo)**.


In [ ]:
%pip install -q langchain langchain-openai langchain-community langchain-groq langchain-huggingface langgraph duckduckgo-search ddgs sympy requests wikipedia-api wikipedia


## 2.- 🔀 Importar librerías

Importamos los elementos comunes que usarás sea cual sea la opción elegida.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from IPython.display import display, Markdown

from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain_core.tools import tool

from sympy import sympify
from langchain_community.tools import DuckDuckGoSearchRun


## 3.- 🧩 Conectar con el LLM (motor del agente)

Usamos la API de **GROQ** con el modelo `gpt-oss-120b`, igual que en los notebooks anteriores. Recuerda indicar tu API key en el bloque del apartado 1.


In [ ]:
# --- Inicializar LLM desde Groq ---
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024,
)


## 4.- 🤖 Repaso: ¿qué es un agente? (patrón ReAct)

Un **agente** es un sistema que percibe su entorno y actúa sobre él para alcanzar un objetivo. Un LLM se convierte en agente cuando le añadimos un **bucle de razonamiento-acción** como el patrón **ReAct** (*Reasoning + Acting*):

<img src="../img/ReAct.webp" style="display: block; margin: auto;">    

```
Observación → Razonamiento → Acción (tool) → Nueva observación → …
```

El ciclo se repite hasta que el agente da la tarea por completada. Las **`tools`** son las *acciones* que el agente puede ejecutar.

<img src="../img/agentic-ai-tool-use-pattern.jpg" style="display: block; margin: auto;">

Recuerda las **3 partes** de una `tool`:

```python
@tool
def nombre_descriptivo(parametros):     # ← 1: nombre claro
    """
    Qué hace la tool.                   # ← 2: docstring (¡lo lee el agente!)
    Args: qué parámetros recibe.
    Returns: qué devuelve (siempre str).
    """
    # lógica de la función             # ← 3: código
    return "resultado como texto"
```


## 5.- 🎯 El ejercicio: elige tu tarea entre 5 opciones

Vas a construir **un agente** que resuelva **una tarea concreta**. Para ello:

1. **Elige UNA** de las 5 opciones que se describen a continuación.
2. Implementa sus **2 `tools` específicas** (apartado 6). Las `tools` comunes (`calculadora` y `busqueda_web`) ya te las damos hechas.
3. Monta el agente con tu lista de `tools` (apartado 7).
4. Pruébalo con las **consultas de ejemplo** de tu opción (apartado 8).

<!-- Nota -->
<div style="background-color: #d1ecf1; color: #000000 ;padding: 15px; border-radius: 5px;">
📌 <b>No hace falta que ejecutes las 5 opciones.</b> Elige una, completa su esqueleto de código y, en el apartado 7, deja en la lista <code>tools</code> sólo las herramientas que hayas definido de verdad.
</div>

---

### 🌍 Opción A — Agente de viajes
**Rol:** ayudar a planificar un viaje.   
**`tools` a implementar:**
- `clima(ciudad)` → devuelve el tiempo actual de una ciudad.
- `cambio_divisa(cantidad, divisa_origen, divisa_destino)` → convierte entre divisas (tabla de tasas fijas).   
**Consultas de ejemplo:** *"Voy a viajar a Londres, ¿qué tiempo hace allí?"*, *"¿Cuántos euros son 250 dólares?"*, *"Busca 3 lugares turísticos de París y dime el tiempo que hace."*

---

### 💪 Opción B — Agente de salud y bienestar
**Rol:** responder dudas de salud y métricas corporales.   
**`tools` a implementar:**
- `imc(peso_kg, altura_m)` → calcula el Índice de Masa Corporal y lo clasifica.
- `conversion_t(celsius)` → convierte ºCelsius a ºFahrenheit.   
**Consultas de ejemplo:** *"Peso 72 kg y mido 1.70 m, ¿cuál es mi IMC y estoy en rango saludable?"*, *"Tengo 38.5 ºC de fiebre, ¿cuánto es en Fahrenheit?"*

---

### 📚 Opción C — Agente tutor educativo
**Rol:** ayudar a estudiar.   
**`tools` a implementar:**
- `buscar_wikipedia(tema)` → consulta información en la Wikipedia (en español).
- `resolver_ecuacion(ecuacion)` → resuelve una ecuación en `x` con `sympy`.   
**Consultas de ejemplo:** *"Explícame quién fue Marie Curie"*, *"Resuelve la ecuación x² − 5x + 6 = 0"*

---

### 💰 Opción D — Agente de finanzas personales
**Rol:** ayudar con cálculos financieros.   
**`tools` a implementar:**
- `cuota_prestamo(capital, interes_anual, anios)` → cuota mensual (sistema francés).
- `interes_compuesto(capital, interes_anual, anios)` → capital final.   
**Consultas de ejemplo:** *"Pido un préstamo de 12.000 € al 6 % anual a 4 años, ¿qué cuota mensual pago?"*, *"Si invierto 1.000 € al 5 % compuesto durante 10 años, ¿cuánto tendré?"*

---

### 🎬 Opción E — Agente de ocio y planes
**Rol:** sugerir planes de tiempo libre.   
**`tools` a implementar:**
- `clima(ciudad)` → devuelve el tiempo actual de una ciudad.
- `plan_aleatorio(categoria)` → sugiere un plan al azar según categoría (cine, deporte, cultura, gastronomia, aire_libre).   
**Consultas de ejemplo:** *"¿Qué tiempo hace en Valencia? Sugéreme un plan al aire libre."*, *"Quiero ver una película, recomiéndame un plan de cine."*


### 🧰 `tools` comunes (ya implementadas para ti)

Estas dos herramientas están disponibles para **todas** las opciones. Ejecuta el bloque y úsalas directamente en tu agente.


In [ ]:
import requests

# -----------------------------------------------
# TOOL común 1: calculadora
# -----------------------------------------------
@tool
def calculadora(expresion: str) -> str:
    """Evalúa una expresión matemática de forma segura. Ejemplos: 'sqrt(144)', '2**10', '3.5*2'."""
    try:
        return str(sympify(expresion))
    except Exception:
        return "Expresión matemática no válida"


# -----------------------------------------------
# TOOL común 2: búsqueda web
# -----------------------------------------------
search = DuckDuckGoSearchRun()

@tool
def busqueda_web(consulta: str) -> str:
    """Busca información actual en Internet (noticias, datos, lugares, etc.)."""
    try:
        return search.run(consulta)
    except Exception as e:
        return f"No se pudo realizar la búsqueda web: {e}"

print("✅ Tools comunes listas: calculadora, busqueda_web")


## 6.- 🔧 Implementa las `tools` de tu opción

Completa **sólo el bloque de la opción que hayas elegido**. Cada bloque es código válido (no falla si lo ejecutas tal cual), pero devuelve `"TODO"` hasta que lo completes: sustituye los `# TODO` y los `[completa...]` por tu implementación real.

> 👉 Recuerda cuidar el **docstring**: es lo que lee el agente para decidir cuándo y cómo usar tu `tool`.


### 🌍 Opción A — Agente de viajes (esqueleto)


In [ ]:
## ===== OPCIÓN A: Agente de viajes =====
## Implementa las 2 tools siguientes.

@tool
def clima(ciudad: str) -> str:
    """[completa: describe qué hace esta tool y qué parámetro recibe]."""
    url = f"https://wttr.in/{ciudad}?format=3"
    response = requests.get(url)
    return response.text  # [completa el return]


@tool
def cambio_divisa(cantidad: float, divisa_origen: str, divisa_destino: str) -> str:
    """[completa: describe qué hace esta tool]."""
    # Tasas de cambio respecto al EURO (1 unidad de divisa = X euros). Amplía si quieres.
    tasas_a_eur = {"EUR": 1.0, "USD": 0.92, "GBP": 1.17, "JPY": 0.0061}
    # TODO: 1) pasa divisa_origen y divisa_destino a mayúsculas
    #       2) convierte 'cantidad' a euros y luego a la divisa destino
    #       3) devuelve un texto del tipo "250 USD = 230.00 EUR"
    return "TODO"


<!-- Desplegable -->
<details>
<summary>💡 Solución Opción A (pulsa para mostrar)</summary>

```python
@tool
def clima(ciudad: str) -> str:
    """Consulta el tiempo/clima actual para una ciudad."""
    url = f"https://wttr.in/{ciudad}?format=3"
    response = requests.get(url)
    return response.text


@tool
def cambio_divisa(cantidad: float, divisa_origen: str, divisa_destino: str) -> str:
    """Convierte una cantidad entre divisas (EUR, USD, GBP, JPY) usando tasas fijas."""
    tasas_a_eur = {"EUR": 1.0, "USD": 0.92, "GBP": 1.17, "JPY": 0.0061}
    origen = divisa_origen.upper()
    destino = divisa_destino.upper()
    if origen not in tasas_a_eur or destino not in tasas_a_eur:
        return f"Divisas no soportadas. Usa: {list(tasas_a_eur.keys())}"
    euros = cantidad * tasas_a_eur[origen]
    resultado = euros / tasas_a_eur[destino]
    return f"{cantidad} {origen} = {resultado:.2f} {destino}"
```
</details>


### 💪 Opción B — Agente de salud y bienestar (esqueleto)


In [ ]:
## ===== OPCIÓN B: Agente de salud y bienestar =====
## Implementa las 2 tools siguientes.

@tool
def imc(peso_kg: float, altura_m: float) -> str:
    """[completa: describe qué hace esta tool]."""
    # TODO: 1) IMC = peso / altura**2
    #       2) clasifica: <18.5 bajo peso | 18.5-24.9 peso saludable | 25-29.9 sobrepeso | >=30 obesidad
    #       3) devuelve un texto con el valor (1 decimal) y la categoría
    return "TODO"


@tool
def conversion_t(celsius: float) -> str:
    """[completa: describe qué hace esta tool]."""
    # TODO: Fahrenheit = celsius * 9/5 + 32 ; devuelve un texto del tipo "38.5°C equivale a 101.3°F"
    return "TODO"


<!-- Desplegable -->
<details>
<summary>💡 Solución Opción B (pulsa para mostrar)</summary>

```python
@tool
def imc(peso_kg: float, altura_m: float) -> str:
    """Calcula el Índice de Masa Corporal (IMC) a partir del peso (kg) y la altura (m), y lo clasifica."""
    valor = peso_kg / (altura_m ** 2)
    if valor < 18.5:
        categoria = "bajo peso"
    elif valor < 25:
        categoria = "peso saludable"
    elif valor < 30:
        categoria = "sobrepeso"
    else:
        categoria = "obesidad"
    return f"Tu IMC es {valor:.1f}, clasificado como {categoria}."


@tool
def conversion_t(celsius: float) -> str:
    """Convierte una temperatura de grados Celsius a grados Fahrenheit."""
    fahrenheit = (celsius * 9 / 5) + 32
    return f"{celsius}°C equivale a {fahrenheit:.1f}°F"
```
</details>


### 📚 Opción C — Agente tutor educativo (esqueleto)

> ⚠️ Esta opción usa la Wikipedia. Asegúrate de haber ejecutado el bloque de instalación (`wikipedia-api`, `wikipedia`).


In [ ]:
## ===== OPCIÓN C: Agente tutor educativo =====
## Implementa las 2 tools siguientes.

from sympy import symbols, solve
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

wikipedia = WikipediaAPIWrapper(lang="es", top_k_results=1)

@tool
def buscar_wikipedia(tema: str) -> str:
    """[completa: describe qué hace esta tool]."""
    # TODO: usa WikipediaQueryRun(api_wrapper=wikipedia).run(tema) y devuelve el resultado
    return "TODO"


@tool
def resolver_ecuacion(ecuacion: str) -> str:
    """[completa: describe qué hace esta tool]. Recibe la expresión igualada a 0, p.ej. 'x**2 - 5*x + 6'."""
    # TODO: x = symbols('x') ; expr = sympify(ecuacion) ; soluciones = solve(expr, x)
    return "TODO"


<!-- Desplegable -->
<details>
<summary>💡 Solución Opción C (pulsa para mostrar)</summary>

```python
@tool
def buscar_wikipedia(tema: str) -> str:
    """Busca información sobre un tema en la Wikipedia (en español)."""
    try:
        wiki_run = WikipediaQueryRun(api_wrapper=wikipedia)
        return wiki_run.run(tema)
    except Exception as e:
        return f"No se pudo consultar Wikipedia: {e}"


@tool
def resolver_ecuacion(ecuacion: str) -> str:
    """Resuelve una ecuación en x. Recibe la expresión igualada a 0, p.ej. 'x**2 - 5*x + 6'."""
    try:
        x = symbols('x')
        expr = sympify(ecuacion)
        soluciones = solve(expr, x)
        return f"Soluciones para x: {soluciones}"
    except Exception as e:
        return f"No se pudo resolver la ecuación: {e}"
```
</details>


### 💰 Opción D — Agente de finanzas personales (esqueleto)


In [ ]:
## ===== OPCIÓN D: Agente de finanzas personales =====
## Implementa las 2 tools siguientes.

@tool
def cuota_prestamo(capital: float, interes_anual: float, anios: float) -> str:
    """[completa: describe qué hace esta tool]."""
    # TODO: r = interes_anual/100/12 ; n = anios*12
    #       cuota = capital * r / (1 - (1+r)**-n)   (si r==0, cuota = capital/n)
    #       devuelve la cuota mensual y el total pagado
    return "TODO"


@tool
def interes_compuesto(capital: float, interes_anual: float, anios: float) -> str:
    """[completa: describe qué hace esta tool]."""
    # TODO: final = capital * (1 + interes_anual/100)**anios ; beneficio = final - capital
    return "TODO"


<!-- Desplegable -->
<details>
<summary>💡 Solución Opción D (pulsa para mostrar)</summary>

```python
@tool
def cuota_prestamo(capital: float, interes_anual: float, anios: float) -> str:
    """Calcula la cuota mensual de un préstamo con el sistema de amortización francés."""
    r = interes_anual / 100 / 12
    n = anios * 12
    if r == 0:
        cuota = capital / n
    else:
        cuota = capital * r / (1 - (1 + r) ** (-n))
    return f"La cuota mensual es {cuota:.2f} EUR durante {n:.0f} meses (total {cuota*n:.2f} EUR)."


@tool
def interes_compuesto(capital: float, interes_anual: float, anios: float) -> str:
    """Calcula el capital final aplicando interés compuesto."""
    final = capital * (1 + interes_anual / 100) ** anios
    beneficio = final - capital
    return f"Tras {anios:.0f} años tendrás {final:.2f} EUR (beneficio: {beneficio:.2f} EUR)."
```
</details>


### 🎬 Opción E — Agente de ocio y planes (esqueleto)


In [ ]:
## ===== OPCIÓN E: Agente de ocio y planes =====
## Implementa las 2 tools siguientes.

import random

@tool
def clima(ciudad: str) -> str:
    """[completa: describe qué hace esta tool]."""
    url = f"https://wttr.in/{ciudad}?format=3"
    return requests.get(url).text  # [completa el return]


@tool
def plan_aleatorio(categoria: str) -> str:
    """[completa: describe qué hace esta tool]. Categorías: cine, deporte, cultura, gastronomia, aire_libre."""
    # TODO: define un diccionario {categoria: [lista de planes]} y devuelve random.choice(...)
    return "TODO"


<!-- Desplegable -->
<details>
<summary>💡 Solución Opción E (pulsa para mostrar)</summary>

```python
@tool
def clima(ciudad: str) -> str:
    """Consulta el tiempo/clima actual para una ciudad."""
    url = f"https://wttr.in/{ciudad}?format=3"
    return requests.get(url).text


@tool
def plan_aleatorio(categoria: str) -> str:
    """Sugiere un plan de ocio al azar según la categoría (cine, deporte, cultura, gastronomia, aire_libre)."""
    planes = {
        "cine": ["maratón de películas en casa", "ver un estreno en el cine", "cinefórum con amigos"],
        "deporte": ["salir a correr", "partido de fútbol con amigos", "sesión de yoga"],
        "cultura": ["visitar un museo", "ruta guiada por el centro", "ir a un concierto"],
        "gastronomia": ["probar un restaurante nuevo", "ruta de tapas", "cocinar una receta nueva"],
        "aire_libre": ["senderismo por la montaña", "picnic en el parque", "paseo en bicicleta"],
    }
    cat = categoria.lower()
    if cat not in planes:
        return f"Categoría no reconocida. Usa: {list(planes.keys())}"
    return f"Plan recomendado ({cat}): {random.choice(planes[cat])}"
```
</details>


## 7.- 🏗️ Construye el agente

Crea la lista `tools` con las **`tools` comunes + las 2 de tu opción** y construye el agente con `create_agent`.

<!-- Nota -->
<div style="background-color: #fff3cd; color: #000000 ;padding: 15px; border-radius: 5px;">
⚠️ Edita la lista <code>tools</code> para incluir <b>sólo</b> las herramientas que hayas implementado de verdad (ejemplo: opción A → <code>clima</code> y <code>cambio_divisa</code>). Si incluyes una <code>tool</code> sin definir obtendrás un <code>NameError</code>.
</div>


In [ ]:
## ===== Monta tu agente =====

# 👇 EDITA esta lista según la opción que hayas elegido.
# Ejemplos:
#   Opción A → [calculadora, busqueda_web, clima, cambio_divisa]
#   Opción B → [calculadora, busqueda_web, imc, conversion_t]
#   Opción C → [calculadora, busqueda_web, buscar_wikipedia, resolver_ecuacion]
#   Opción D → [calculadora, busqueda_web, cuota_prestamo, interes_compuesto]
#   Opción E → [calculadora, busqueda_web, clima, plan_aleatorio]
tools = [calculadora, busqueda_web]  # <-- añade aquí tus 2 tools

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "Eres un asistente útil. Usa las herramientas de forma secuencial cuando sea necesario, "
        "esperando el resultado de cada una antes de llamar a la siguiente. Responde siempre en español."
    ),
)

print(f"✅ Agente creado con {len(tools)} tools: {[t.name for t in tools]}")


## 8.- 🔍 Prueba tu agente

Lanza una consulta relacionada con tu opción y muestra la respuesta. Prueba a escribir varias para comprobar que el agente elige bien las `tools`.


In [ ]:
# --- Ejecutar la consulta ---
pregunta = "Escribe aquí una consulta propia de tu opción"  # 👈 cámbiala
# Ejemplos:
#   A) "Voy a viajar a Londres, ¿qué tiempo hace allí y cuántos euros son 250 dólares?"
#   B) "Peso 72 kg y mido 1.70 m, ¿cuál es mi IMC? Convierte 38.5 ºC a Fahrenheit."
#   C) "Explícame quién fue Marie Curie y resuelve la ecuación x**2 - 5*x + 6 = 0"
#   D) "Préstamo de 12000 € al 6 % anual a 4 años: ¿cuota mensual? E invierte 1000 € al 5 % a 10 años."
#   E) "¿Qué tiempo hace en Valencia? Sugéreme un plan al aire libre."

response = agent.invoke({"messages": [("user", pregunta)]})
display(Markdown(response['messages'][-1].content))


### 🔎 Inspecciona el proceso de razonamiento

El siguiente bloque muestra **todo el intercambio**: mensajes del usuario, decisiones de la IA, llamadas a `tools` y consumo de tokens. Así puedes comprobar *qué herramienta usó el agente y por qué*.


In [ ]:
# --- Visualizar lista de mensajes completa para ver el proceso de pensamiento ---
for msg in response['messages']:
    tipo = type(msg).__name__

    if tipo == "HumanMessage":
        print(f"👤 USUARIO: {msg.content}\n")

    elif tipo == "AIMessage":
        print("🤖 IA:")
        if msg.content:
            display(Markdown(msg.content))
        if msg.tool_calls:
            print("  🔧 Herramientas llamadas:")
            for tc in msg.tool_calls:
                print(f"     - {tc['name']}({tc['args']})")
        tokens = msg.usage_metadata
        if tokens:
            print(f"  📊 Tokens: entrada={tokens['input_tokens']}, salida={tokens['output_tokens']}, total={tokens['total_tokens']}")
        print()

    elif tipo == "ToolMessage":
        print(f"  🛠️  TOOL [{msg.name}]: {msg.content}\n")


## 9.- 🧠 Reto extra (opcional): historial de conversación

Configura un **historial de mensajes** para que tu agente mantenga el contexto entre consultas, igual que en el notebook 0. Lanza una primera consulta, guarda el historial y haz una segunda pregunta relacionada.


In [ ]:
## ===== Reto extra: historial de conversación =====
from langchain_core.messages import HumanMessage, AIMessage

# 1) Primera consulta y guardado del historial
response = agent.invoke({"messages": [("user", pregunta)]})
message_history = response["messages"]

# 2) Segunda consulta (apóyate en el contexto anterior)
new_query = "Escribe aquí una segunda consulta relacionada"  # 👈 cámbiala
messages = agent.invoke({"messages": message_history + [("user", new_query)]})

# 3) Mostrar sólo los mensajes relevantes
filtered = [m for m in messages["messages"]
            if isinstance(m, (HumanMessage, AIMessage)) and m.content.strip()]
display(Markdown(messages['messages'][-1].content))


## 10.- ✅ Rúbrica de evaluación

Marca los criterios cumplidos en tu entrega:

| # | Criterio | ✔️ |
|---|----------|----|
| 1 | Has elegido **una** opción y la has indicado claramente. | |
| 2 | Has implementado **2 `tools` específicas** con el decorador `@tool`. | |
| 3 | Cada `tool` tiene **nombre descriptivo**, **docstring** útil y **devuelve un `str`**. | |
| 4 | Has creado el agente con `create_agent`, pasando `tools` y un `system_prompt`. | |
| 5 | El agente **responde correctamente** a las consultas de ejemplo de tu opción. | |
| 6 | Has **inspeccionado el razonamiento** (qué `tools` se llaman y en qué orden). | |
| 7 | *(Extra)* Has añadido **historial de conversación**. | |
| 8 | *(Extra)* Has añadido una **tercera `tool`** propia o mejorado una existente. | |

<!-- Nota -->
<div style="background-color: #d4edda; color: #000000 ;padding: 15px; border-radius: 5px;">
🎓 <b>¿Qué se valora?</b> La claridad del <i>docstring</i>, que las <i>tools</i> devuelvan siempre texto útil, que el agente sepa encadenar varias herramientas y la calidad/legibilidad de las respuestas. ¡Y que funcione! 🚀
</div>

------

**Fin del ejercicio.** ¡Buen trabajo construyendo tu primer agente con herramientas! 🤖🔧
